# Inviscid Burgers equation on a finite interval: CPU implementation

This notebook solves the one-dimensional inviscid Burgers equation

$$
u_t+\left(\frac{u^2}{2}\right)_x=0, \qquad x\in[0,2],
$$

with the initial and boundary data used in the chapter. It implements the
conservative MacCormack predictor--corrector method, compares the numerical
solution with the exact solution, and follows the nonlinear deformation up to
the breaking time. A few post-breaking computations are included only to show
the limitations of this smooth-solution method near a shock.

The notebook is organized as a small reusable module. The final sections
contain complete examples that can be launched with a single function call.


## 1. Mathematical problem

The test problem is

$$
u_t+f(u)_x=0, \qquad f(u)=\frac{u^2}{2}, \qquad 0\le x\le2,
$$

with

$$
u(x,0)=
\begin{cases}
1-x, & 0\le x\le1,\\
0, & 1<x\le2,
\end{cases}
\qquad
u(0,t)=1, \qquad u(2,t)=0.
$$

Before the breaking time $t_{\mathrm b}=1$, the exact solution is

$$
u(x,t)=
\begin{cases}
1, & 0\le x<t,\\
\dfrac{1-x}{1-t}, & t\le x\le1,\\
0, & 1<x\le2.
\end{cases}
$$

At $t=1$ the decreasing ramp becomes vertical. For $t\ge1$, the entropy
solution contains a shock at $x_s(t)=(1+t)/2$. The MacCormack method used here
is intended for the smooth regime; the entropy solution is retained after
breaking only as a reference for illustrating why a dedicated shock-capturing
method is then needed.


## 2. Imports and result container


In [1]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import numpy as np

Array = np.ndarray


`BurgersRunResult` collects the final numerical and exact fields, sampled
diagnostic histories, and effective discretization parameters. When
`store_snapshots=True`, `snapshot_times` and `snapshots` also contain the data
needed to build an animation. Ordinary simulations leave them equal to
`None`, avoiding unnecessary memory use.

The reported relative error uses the discrete $L^2$ norm. The mass error
compares the numerical integral of $u$ with the integral of the exact
solution at the same time.


In [2]:
@dataclass(frozen=True)
class BurgersRunResult:
    """Numerical solution and diagnostics from one Burgers run."""

    x: Array
    numerical: Array
    exact: Array
    times: Array
    error_history: Array
    mass_error_history: Array
    minimum_history: Array
    maximum_history: Array
    dx: float
    dt: float
    nsteps: int
    courant: float
    relative_l2_error: float
    mass_error: float
    minimum: float
    maximum: float
    snapshot_times: Array | None = None
    snapshots: Array | None = None


## 3. Initial data, boundary data, and exact solution


In [3]:
def flux(u: Array) -> Array:
    """Return the Burgers flux f(u)=u^2/2."""
    return 0.5 * u * u


def initial_condition(x: Array) -> Array:
    """Return the piecewise-linear initial condition used in the chapter."""
    return np.where(x <= 1.0, 1.0 - x, 0.0)


In [4]:
def inflow_value(time: float) -> float:
    """Return the prescribed value at the left boundary x=0."""
    del time
    return 1.0


def outflow_value(time: float) -> float:
    """Return the right-endpoint value used by this particular test."""
    del time
    return 0.0


def exact_entropy_solution(x: Array, time: float) -> Array:
    """Evaluate the exact entropy solution of the selected test problem."""
    if time < 0.0:
        raise ValueError("time must be non-negative")

    if time < 1.0:
        exact = np.zeros_like(x, dtype=np.float64)
        exact[x < time] = 1.0
        ramp = (x >= time) & (x <= 1.0)
        exact[ramp] = (1.0 - x[ramp]) / (1.0 - time)
        return exact

    shock_position = 0.5 * (1.0 + time)
    return np.where(x < shock_position, 1.0, 0.0)


## 4. Boundary treatment

For this test, the exact solution satisfies $0\le u\le1$, so characteristics
travel to the right. The left endpoint is an inflow boundary and receives the
physical value $u(0,t)=1$. The value $u(2,t)=0$ is an ad hoc right closure that
is compatible with the exact solution throughout the simulated interval; it
is not intended as a general outflow treatment.

The forward predictor is evaluated for $i=0,\ldots,N-1$ and needs one
provisional value at the last node, which is set to zero. The backward
corrector is evaluated only for $i=1,\ldots,N-1$. The two endpoint values of
the corrected field are then restored explicitly.


In [5]:
def close_predictor(predicted: Array, time: float) -> None:
    """Supply the last provisional value required by the test stencil."""
    predicted[-1] = outflow_value(time)


def impose_corrected_boundaries(field: Array, time: float) -> None:
    """Impose the two endpoint values after the corrector stage."""
    field[0] = inflow_value(time)
    field[-1] = outflow_value(time)


## 5. Conservative MacCormack scheme

The method uses the same predictor--corrector organization introduced for
linear advection. Only the flux changes: the linear flux $au$ is replaced by
$f(u)=u^2/2$. With $\lambda=\Delta t/\Delta x$, the forward predictor is

$$
\widetilde u_i=u_i^n-\lambda
\left[f(u_{i+1}^n)-f(u_i^n)\right],
$$

and the backward corrector is

$$
u_i^{n+1}=\frac12\left\{u_i^n+\widetilde u_i
-\lambda\left[f(\widetilde u_i)-f(\widetilde u_{i-1})\right]\right\}.
$$

The predicted values remain on the original spatial grid and approximate the
solution after a complete time step; this is MacCormack, not the staggered
half-step construction of the two-step Richtmyer method. For smooth solutions
the method is second-order accurate in space and time. Its stability condition
depends on the evolving characteristic speed $f'(u)=u$:

$$
\max_i |u_i^n|\,\frac{\Delta t}{\Delta x}\le C, \qquad C<1.
$$

For the present test $\max|u|=1$, so a fixed step based on unit maximum speed
is sufficient. The two array sweeps also map directly to two successive GPU
kernels in the implementation developed later in the chapter.


In [6]:
def maccormack_step(old: Array, *, lam: float, time: float, dt: float) -> Array:
    """Advance one conservative MacCormack time step."""
    predicted = np.empty_like(old)
    new = np.empty_like(old)

    # Predictor: forward difference of the physical flux.
    predicted[:-1] = old[:-1] - lam * (flux(old[1:]) - flux(old[:-1]))
    close_predictor(predicted, time + dt)

    # Corrector: backward difference of the predicted flux.
    new[1:-1] = 0.5 * (
        old[1:-1]
        + predicted[1:-1]
        - lam * (flux(predicted[1:-1]) - flux(predicted[:-2]))
    )
    impose_corrected_boundaries(new, time + dt)
    return new


## 6. Solver


In [7]:
def _discrete_l2(values: Array, dx: float) -> float:
    """Return the uniform-grid discrete L2 norm."""
    return float(np.sqrt(dx) * np.linalg.norm(values))


def _discrete_mass(values: Array, dx: float) -> float:
    """Return the composite-trapezoidal integral on a uniform grid."""
    return float(dx * (0.5 * values[0] + values[1:-1].sum() + 0.5 * values[-1]))


def solve_burgers_maccormack(
    *,
    nx: int = 400,
    cfl: float = 0.4,
    final_time: float = 1.10,
    length: float = 2.0,
    save_every: int = 1,
    store_snapshots: bool = False,
    snapshot_every: int = 1,
) -> BurgersRunResult:
    """Solve the chapter test with the conservative MacCormack method.

    ``nx`` is the number of cells, so the returned grid contains ``nx+1``
    points. The requested CFL value defines a tentative time step using the
    known maximum speed one. The number of steps is rounded up and ``dt`` is
    reduced so that the last step lands exactly on ``final_time``.

    Diagnostics are sampled every ``save_every`` steps and always at final
    time. Full fields are retained only when ``store_snapshots=True``.
    """
    if nx < 8:
        raise ValueError("nx must be at least 8")
    if length != 2.0:
        raise ValueError("This exact test problem is defined on [0, 2]")
    if final_time <= 0.0:
        raise ValueError("final_time must be positive")
    if not (0.0 < cfl <= 1.0):
        raise ValueError("cfl must satisfy 0 < cfl <= 1")
    if save_every < 1:
        raise ValueError("save_every must be positive")
    if snapshot_every < 1:
        raise ValueError("snapshot_every must be positive")

    dx = length / nx
    tentative_dt = cfl * dx  # max |u| = 1 for this test problem
    nsteps = max(1, int(np.ceil(final_time / tentative_dt)))
    dt = final_time / nsteps
    courant = dt / dx
    lam = dt / dx
    x = np.linspace(0.0, length, nx + 1, dtype=np.float64)

    numerical = initial_condition(x)
    exact_initial = exact_entropy_solution(x, 0.0)
    initial_norm = max(_discrete_l2(exact_initial, dx), 1.0e-30)

    times = [0.0]
    error_history = [0.0]
    mass_error_history = [
        _discrete_mass(numerical, dx) - _discrete_mass(exact_initial, dx)
    ]
    minimum_history = [float(np.min(numerical))]
    maximum_history = [float(np.max(numerical))]
    snapshot_times = [0.0] if store_snapshots else []
    snapshots = [numerical.copy()] if store_snapshots else []

    def record(field: Array, step: int) -> None:
        time_now = step * dt
        exact_now = exact_entropy_solution(x, time_now)
        times.append(time_now)
        error_history.append(_discrete_l2(field - exact_now, dx) / initial_norm)
        mass_error_history.append(
            _discrete_mass(field, dx) - _discrete_mass(exact_now, dx)
        )
        minimum_history.append(float(np.min(field)))
        maximum_history.append(float(np.max(field)))

    def record_snapshot(field: Array, step: int) -> None:
        if store_snapshots and (step % snapshot_every == 0 or step == nsteps):
            snapshot_times.append(step * dt)
            snapshots.append(field.copy())

    for step in range(nsteps):
        numerical = maccormack_step(
            numerical, lam=lam, time=step * dt, dt=dt
        )
        accepted_step = step + 1
        if accepted_step % save_every == 0 or accepted_step == nsteps:
            record(numerical, accepted_step)
        record_snapshot(numerical, accepted_step)

    exact = exact_entropy_solution(x, final_time)
    relative_l2_error = _discrete_l2(numerical - exact, dx) / initial_norm
    mass_error = _discrete_mass(numerical, dx) - _discrete_mass(exact, dx)

    return BurgersRunResult(
        x=x,
        numerical=numerical,
        exact=exact,
        times=np.asarray(times, dtype=np.float64),
        error_history=np.asarray(error_history, dtype=np.float64),
        mass_error_history=np.asarray(mass_error_history, dtype=np.float64),
        minimum_history=np.asarray(minimum_history, dtype=np.float64),
        maximum_history=np.asarray(maximum_history, dtype=np.float64),
        dx=dx,
        dt=dt,
        nsteps=nsteps,
        courant=courant,
        relative_l2_error=float(relative_l2_error),
        mass_error=float(mass_error),
        minimum=float(np.min(numerical)),
        maximum=float(np.max(numerical)),
        snapshot_times=(
            np.asarray(snapshot_times, dtype=np.float64)
            if store_snapshots else None
        ),
        snapshots=(np.stack(snapshots) if store_snapshots else None),
    )


## 7. Reporting and plotting helpers


In [8]:
def print_run_summary(result: BurgersRunResult, *, nx: int, final_time: float) -> None:
    """Print a compact, consistently formatted summary of one run."""
    regime = "smooth" if final_time < 1.0 else "post-breaking reference"
    print("method                  = conservative MacCormack")
    print(f"regime                  = {regime}")
    print(f"grid cells              = {nx}")
    print(f"time steps              = {result.nsteps}")
    print(f"dt                      = {result.dt:.12e}")
    print(f"max-speed Courant no.   = {result.courant:.12e}")
    print(f"relative L2 error       = {result.relative_l2_error:.12e}")
    print(f"mass error              = {result.mass_error:.12e}")
    print(f"numerical minimum       = {result.minimum:.12e}")
    print(f"numerical maximum       = {result.maximum:.12e}")


def plot_final_solution(
    result: BurgersRunResult,
    *,
    title: str | None = None,
    output: Path | str | None = None,
    show: bool = True,
):
    """Plot the final numerical and exact solutions and return the figure."""
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    ax.plot(result.x, result.exact, "k--", linewidth=2.2, label="Exact")
    ax.plot(result.x, result.numerical, linewidth=1.6, label="MacCormack")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    if title is not None:
        ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    if output is not None:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180)
    if show:
        plt.show()
    return fig


## 8. Complete example functions

`run_single_example` performs one simulation and plots its final state.
`run_snapshot_example` shows nonlinear steepening at selected times.
`run_resolution_comparison_example` repeats the same physical problem on
several grids. These functions play the same organizational role as the
single-run, closure-study, and method-comparison helpers in the advection
notebook, while using comparisons that are meaningful for the one MacCormack
scheme studied here.


In [9]:
def run_single_example(
    *,
    nx: int = 400,
    cfl: float = 0.4,
    final_time: float = 1.10,
    length: float = 2.0,
    save_every: int = 1,
    output: Path | str | None = None,
    show: bool = True,
) -> BurgersRunResult:
    """Run one complete simulation, print diagnostics, and make a plot."""
    result = solve_burgers_maccormack(
        nx=nx,
        cfl=cfl,
        final_time=final_time,
        length=length,
        save_every=save_every,
    )
    print_run_summary(result, nx=nx, final_time=final_time)
    plot_final_solution(
        result,
        title=f"Inviscid Burgers equation at t={final_time:g}",
        output=output,
        show=show,
    )
    return result


def run_snapshot_example(
    *,
    nx: int = 400,
    cfl: float = 0.4,
    final_time: float = 1.10,
    length: float = 2.0,
    snapshots_requested: int = 7,
    output: Path | str | None = None,
    show: bool = True,
) -> BurgersRunResult:
    """Plot sampled fields to display nonlinear deformation and steepening."""
    if snapshots_requested < 2:
        raise ValueError("snapshots_requested must be at least 2")

    estimated_steps = max(1, int(np.ceil(final_time / (cfl * length / nx))))
    snapshot_every = max(
        1, int(np.ceil(estimated_steps / (snapshots_requested - 1)))
    )
    result = solve_burgers_maccormack(
        nx=nx,
        cfl=cfl,
        final_time=final_time,
        length=length,
        save_every=snapshot_every,
        store_snapshots=True,
        snapshot_every=snapshot_every,
    )
    if result.snapshots is None or result.snapshot_times is None:
        raise RuntimeError("Snapshot storage was not initialized")

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    for time_now, field in zip(result.snapshot_times, result.snapshots):
        ax.plot(result.x, field, label=fr"$t={time_now:.2f}$")
    ax.set_title("Nonlinear steepening and the post-breaking limitation")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()

    if output is not None:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180)
    if show:
        plt.show()
    return result


def run_resolution_comparison_example(
    *,
    grids: Sequence[int] = (100, 200, 400),
    cfl: float = 0.4,
    final_time: float = 0.8,
    length: float = 2.0,
    output: Path | str | None = None,
    show: bool = True,
) -> dict[int, BurgersRunResult]:
    """Compare MacCormack solutions at one smooth time on several grids."""
    results = {
        int(nx): solve_burgers_maccormack(
            nx=int(nx), cfl=cfl, final_time=final_time, length=length
        )
        for nx in grids
    }

    print("Grid-resolution comparison:")
    for nx, result in results.items():
        print(f"  nx={nx:4d}: relative L2 error = {result.relative_l2_error:.6e}")

    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    reference = results[next(iter(results))]
    ax.plot(reference.x, reference.exact, "k--", linewidth=2.2, label="Exact")
    for nx, result in results.items():
        ax.plot(result.x, result.numerical, label=fr"$N_x={nx}$")
    ax.set_title(f"Grid comparison at t={final_time:g}")
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()

    if output is not None:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180)
    if show:
        plt.show()
    return results


def generate_burgers_figure(output: Path | str) -> BurgersRunResult:
    """Backward-compatible wrapper that saves the snapshot figure."""
    return run_snapshot_example(output=output, show=False)


## 9. Animation and video download

The functions below rerun the solver with snapshot storage enabled, encode MP4
animations with Matplotlib and `ffmpeg`, and optionally display or download
the result. In Google Colab, `download=True` opens the browser download dialog.
In ordinary Jupyter, the notebook displays a file link instead.

`max_frames` changes only how often computed fields are copied for the video;
it does not change the numerical time step. Comparison videos synchronize runs
with different grids or CFL numbers at common physical times.


In [10]:
def _running_in_colab() -> bool:
    """Return True when the current kernel is hosted by Google Colab."""
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def ensure_ffmpeg() -> None:
    """Ensure that Matplotlib can access an ffmpeg MP4 writer."""
    import shutil
    import subprocess

    if shutil.which("ffmpeg") is not None:
        return
    if not _running_in_colab():
        raise RuntimeError(
            "ffmpeg was not found. Install ffmpeg or run the notebook in "
            "Google Colab before requesting an MP4 file."
        )

    print("ffmpeg not found; installing it in the Colab runtime...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("The ffmpeg installation did not complete successfully")


def download_generated_file(path: Path | str) -> Path:
    """Download a generated file in Colab or display a Jupyter file link."""
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(path)

    if _running_in_colab():
        from google.colab import files

        files.download(str(path))
    else:
        from IPython.display import FileLink, display as ipy_display

        print("Outside Google Colab: use the link below to download the file.")
        ipy_display(FileLink(str(path)))
    return path


def create_solution_animation(
    result: BurgersRunResult,
    *,
    output: Path | str = "burgers_steepening.mp4",
    fps: int = 25,
    dpi: int = 120,
    preview: bool = True,
    dynamic_ylim: bool = False,
) -> Path:
    """Create, save, and optionally display one solution animation."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FFMpegWriter, FuncAnimation

    if result.snapshots is None or result.snapshot_times is None:
        raise ValueError(
            "No snapshots are available. Run solve_burgers_maccormack with "
            "store_snapshots=True."
        )
    if fps < 1:
        raise ValueError("fps must be positive")
    if dpi < 40:
        raise ValueError("dpi must be at least 40")

    ensure_ffmpeg()
    output = Path(output)
    if output.suffix.lower() != ".mp4":
        raise ValueError("The animation output filename must end in .mp4")
    output.parent.mkdir(parents=True, exist_ok=True)

    snapshots = result.snapshots
    snapshot_times = result.snapshot_times
    data_min = min(float(np.min(snapshots)), 0.0)
    data_max = max(float(np.max(snapshots)), 1.0)
    padding = max(0.08 * (data_max - data_min), 0.05)
    initial_norm = max(_discrete_l2(snapshots[0], result.dx), 1.0e-30)

    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    numerical_line, = ax.plot([], [], linewidth=2.0, label="MacCormack")
    exact_line, = ax.plot([], [], "k--", linewidth=1.8, label="Exact")
    time_text = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top")
    diagnostic_text = ax.text(0.02, 0.88, "", transform=ax.transAxes, va="top")
    ax.set_xlim(float(result.x[0]), float(result.x[-1]))
    ax.set_ylim(data_min - padding, data_max + padding)
    ax.set_xlabel("x")
    ax.set_ylabel("u")
    ax.set_title(f"Inviscid Burgers equation; C={result.courant:.3f}")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")
    fig.tight_layout()

    def initialize():
        numerical_line.set_data([], [])
        exact_line.set_data([], [])
        time_text.set_text("")
        diagnostic_text.set_text("")
        return numerical_line, exact_line, time_text, diagnostic_text

    def update(frame: int):
        time_now = float(snapshot_times[frame])
        numerical_now = snapshots[frame]
        exact_now = exact_entropy_solution(result.x, time_now)
        error = _discrete_l2(numerical_now - exact_now, result.dx) / initial_norm
        numerical_line.set_data(result.x, numerical_now)
        exact_line.set_data(result.x, exact_now)
        time_text.set_text(f"t = {time_now:.4f}")
        diagnostic_text.set_text(
            f"relative L2 error = {error:.3e}\n"
            f"range = [{np.min(numerical_now):.3e}, {np.max(numerical_now):.3e}]"
        )
        if dynamic_ylim:
            frame_min = min(float(np.min(numerical_now)), 0.0)
            frame_max = max(float(np.max(numerical_now)), 1.0)
            frame_padding = max(0.08 * (frame_max - frame_min), 0.05)
            ax.set_ylim(frame_min - frame_padding, frame_max + frame_padding)
        return numerical_line, exact_line, time_text, diagnostic_text

    animation = FuncAnimation(
        fig,
        update,
        frames=len(snapshot_times),
        init_func=initialize,
        interval=1000.0 / fps,
        blit=not dynamic_ylim,
    )
    writer = FFMpegWriter(
        fps=fps,
        bitrate=1800,
        metadata={"title": "Inviscid Burgers equation: MacCormack"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)

    print(
        f"Video saved to {output.resolve()} "
        f"({len(snapshot_times)} frames, {fps} fps)."
    )
    if preview:
        from IPython.display import Video, display as ipy_display

        ipy_display(Video(str(output), embed=True))
    return output.resolve()


def run_animation_example(
    *,
    nx: int = 400,
    cfl: float = 0.4,
    final_time: float = 1.10,
    length: float = 2.0,
    max_frames: int = 180,
    fps: int = 25,
    dpi: int = 120,
    output: Path | str = "burgers_steepening.mp4",
    preview: bool = True,
    download: bool = False,
    dynamic_ylim: bool = False,
) -> tuple[BurgersRunResult, Path]:
    """Run one simulation and produce a downloadable propagation video."""
    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")

    estimated_steps = max(1, int(np.ceil(final_time / (cfl * length / nx))))
    snapshot_every = max(1, int(np.ceil(estimated_steps / (max_frames - 1))))
    result = solve_burgers_maccormack(
        nx=nx,
        cfl=cfl,
        final_time=final_time,
        length=length,
        save_every=snapshot_every,
        store_snapshots=True,
        snapshot_every=snapshot_every,
    )
    print_run_summary(result, nx=nx, final_time=final_time)
    video_path = create_solution_animation(
        result,
        output=output,
        fps=fps,
        dpi=dpi,
        preview=preview,
        dynamic_ylim=dynamic_ylim,
    )
    if download:
        download_generated_file(video_path)
    return result, video_path


def _interpolate_snapshot_history(
    result: BurgersRunResult, target_times: Array
) -> Array:
    """Linearly interpolate stored fields onto common animation times."""
    if result.snapshots is None or result.snapshot_times is None:
        raise ValueError("The result does not contain animation snapshots")
    times = result.snapshot_times
    if times.size < 2:
        raise ValueError("At least two snapshots are required")
    if target_times[0] < times[0] or target_times[-1] > times[-1]:
        raise ValueError("Target animation times lie outside the stored history")

    right = np.searchsorted(times, target_times, side="right")
    right = np.clip(right, 1, times.size - 1)
    left = right - 1
    denominator = times[right] - times[left]
    weight = (target_times - times[left]) / denominator
    return (
        (1.0 - weight)[:, None] * result.snapshots[left]
        + weight[:, None] * result.snapshots[right]
    )


def create_parameter_comparison_animation(
    baseline: BurgersRunResult,
    comparison: BurgersRunResult,
    *,
    baseline_label: str,
    comparison_label: str,
    output: Path | str,
    max_frames: int = 90,
    fps: int = 20,
    dpi: int = 100,
    preview: bool = True,
) -> Path:
    """Create a synchronized side-by-side parameter comparison."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FFMpegWriter, FuncAnimation

    if max_frames < 2:
        raise ValueError("max_frames must be at least 2")
    if baseline.snapshot_times is None or comparison.snapshot_times is None:
        raise ValueError("Both simulations must contain snapshots")

    ensure_ffmpeg()
    output = Path(output)
    if output.suffix.lower() != ".mp4":
        raise ValueError("The animation output filename must end in .mp4")
    output.parent.mkdir(parents=True, exist_ok=True)

    final_time = min(
        float(baseline.snapshot_times[-1]),
        float(comparison.snapshot_times[-1]),
    )
    frame_times = np.linspace(0.0, final_time, max_frames)
    baseline_frames = _interpolate_snapshot_history(baseline, frame_times)
    comparison_frames = _interpolate_snapshot_history(comparison, frame_times)
    grids = (baseline.x, comparison.x)
    frames = (baseline_frames, comparison_frames)
    norms = (
        max(_discrete_l2(baseline_frames[0], baseline.dx), 1.0e-30),
        max(_discrete_l2(comparison_frames[0], comparison.dx), 1.0e-30),
    )
    dx_values = (baseline.dx, comparison.dx)

    all_values = np.concatenate([baseline_frames.ravel(), comparison_frames.ravel()])
    data_min = min(float(np.min(all_values)), 0.0)
    data_max = max(float(np.max(all_values)), 1.0)
    padding = max(0.08 * (data_max - data_min), 0.05)

    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.6), sharey=True)
    numerical_lines = []
    exact_lines = []
    time_texts = []
    error_texts = []
    for ax, label in zip(axes, (baseline_label, comparison_label)):
        numerical_line, = ax.plot([], [], linewidth=2.0, label="MacCormack")
        exact_line, = ax.plot([], [], "k--", linewidth=1.8, label="Exact")
        time_text = ax.text(0.02, 0.96, "", transform=ax.transAxes, va="top")
        error_text = ax.text(0.02, 0.88, "", transform=ax.transAxes, va="top")
        ax.set_xlim(0.0, 2.0)
        ax.set_ylim(data_min - padding, data_max + padding)
        ax.set_xlabel("x")
        ax.set_title(label)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper right")
        numerical_lines.append(numerical_line)
        exact_lines.append(exact_line)
        time_texts.append(time_text)
        error_texts.append(error_text)
    axes[0].set_ylabel("u")
    fig.suptitle("Conservative MacCormack parameter comparison")
    fig.tight_layout()

    def initialize():
        artists = []
        for numerical_line, exact_line, time_text, error_text in zip(
            numerical_lines, exact_lines, time_texts, error_texts
        ):
            numerical_line.set_data([], [])
            exact_line.set_data([], [])
            time_text.set_text("")
            error_text.set_text("")
            artists.extend([numerical_line, exact_line, time_text, error_text])
        return tuple(artists)

    def update(frame: int):
        artists = []
        time_now = float(frame_times[frame])
        for index in range(2):
            exact_now = exact_entropy_solution(grids[index], time_now)
            numerical_now = frames[index][frame]
            error = _discrete_l2(
                numerical_now - exact_now, dx_values[index]
            ) / norms[index]
            numerical_lines[index].set_data(grids[index], numerical_now)
            exact_lines[index].set_data(grids[index], exact_now)
            time_texts[index].set_text(f"t = {time_now:.4f}")
            error_texts[index].set_text(f"relative L2 error = {error:.3e}")
            artists.extend([
                numerical_lines[index],
                exact_lines[index],
                time_texts[index],
                error_texts[index],
            ])
        return tuple(artists)

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frame_times),
        init_func=initialize,
        interval=1000.0 / fps,
        blit=True,
    )
    writer = FFMpegWriter(
        fps=fps,
        bitrate=2400,
        metadata={"title": "Burgers MacCormack parameter comparison"},
    )
    animation.save(str(output), writer=writer, dpi=dpi)
    plt.close(fig)

    print(
        f"Video saved to {output.resolve()} "
        f"({len(frame_times)} synchronized frames, {fps} fps)."
    )
    if preview:
        from IPython.display import Video, display as ipy_display

        ipy_display(Video(str(output), embed=True))
    return output.resolve()


def run_parameter_comparison_example(
    *,
    baseline_parameters: dict[str, object],
    comparison_parameters: dict[str, object],
    baseline_label: str,
    comparison_label: str,
    output: Path | str,
    max_frames: int = 90,
    fps: int = 20,
    dpi: int = 100,
    preview: bool = True,
    download: bool = False,
) -> tuple[dict[str, BurgersRunResult], Path]:
    """Run two parameter choices and animate their synchronized comparison."""
    defaults: dict[str, object] = {
        "nx": 100,
        "cfl": 0.4,
        "final_time": 0.8,
        "length": 2.0,
    }
    baseline_config = defaults | baseline_parameters
    comparison_config = defaults | comparison_parameters
    if baseline_config["final_time"] != comparison_config["final_time"]:
        raise ValueError("Both comparison runs must use the same final_time")

    def solve_config(config: dict[str, object]) -> BurgersRunResult:
        nx = int(config["nx"])
        cfl = float(config["cfl"])
        final_time = float(config["final_time"])
        length = float(config["length"])
        estimated_steps = max(
            1, int(np.ceil(final_time / (cfl * length / nx)))
        )
        snapshot_every = max(
            1, int(np.ceil(estimated_steps / (max_frames - 1)))
        )
        return solve_burgers_maccormack(
            nx=nx,
            cfl=cfl,
            final_time=final_time,
            length=length,
            save_every=snapshot_every,
            store_snapshots=True,
            snapshot_every=snapshot_every,
        )

    baseline = solve_config(baseline_config)
    comparison = solve_config(comparison_config)
    print(
        f"{baseline_label}: final relative L2 error = "
        f"{baseline.relative_l2_error:.6e}"
    )
    print(
        f"{comparison_label}: final relative L2 error = "
        f"{comparison.relative_l2_error:.6e}"
    )
    video_path = create_parameter_comparison_animation(
        baseline,
        comparison,
        baseline_label=baseline_label,
        comparison_label=comparison_label,
        output=output,
        max_frames=max_frames,
        fps=fps,
        dpi=dpi,
        preview=preview,
    )
    if download:
        download_generated_file(video_path)
    return {"baseline": baseline, "comparison": comparison}, video_path


def run_validation_campaign(
    *,
    output_directory: Path | str = "burgers_validation_videos",
    preview_each: bool = False,
    download_archive: bool = False,
    fps: int = 20,
    dpi: int = 100,
) -> tuple[dict[str, object], dict[str, Path], Path]:
    """Generate ten animated validation cases and package them in one ZIP."""
    import zipfile

    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    results: dict[str, object] = {}
    videos: dict[str, Path] = {}

    single_cases = (
        ("01_early_smooth_transport", 0.40, 160, 0.4),
        ("02_nonlinear_deformation", 0.70, 160, 0.4),
        ("03_strong_steepening", 0.90, 160, 0.4),
        ("04_near_breaking", 0.99, 200, 0.4),
        ("05_post_breaking_limitation", 1.08, 200, 0.4),
        ("06_reduced_cfl_near_breaking", 0.99, 200, 0.2),
    )
    for case_name, final_time, nx, cfl in single_cases:
        print(f"\nGenerating {case_name}.mp4")
        result, video = run_animation_example(
            nx=nx,
            cfl=cfl,
            final_time=final_time,
            max_frames=90,
            fps=fps,
            dpi=dpi,
            output=output_directory / f"{case_name}.mp4",
            preview=preview_each,
            download=False,
            dynamic_ylim=final_time >= 1.0,
        )
        results[case_name] = result
        videos[case_name] = video

    comparisons = (
        (
            "07_smooth_grid_refinement",
            {"nx": 100, "cfl": 0.4, "final_time": 0.70},
            {"nx": 400, "cfl": 0.4, "final_time": 0.70},
            r"Baseline: $N_x=100$",
            r"Refined: $N_x=400$",
        ),
        (
            "08_cfl_comparison",
            {"nx": 200, "cfl": 0.4, "final_time": 0.90},
            {"nx": 200, "cfl": 0.2, "final_time": 0.90},
            r"Baseline: $C=0.4$",
            r"Smaller step: $C=0.2$",
        ),
        (
            "09_near_breaking_refinement",
            {"nx": 100, "cfl": 0.4, "final_time": 0.98},
            {"nx": 400, "cfl": 0.4, "final_time": 0.98},
            r"Baseline: $N_x=100$",
            r"Refined: $N_x=400$",
        ),
        (
            "10_post_breaking_refinement_limit",
            {"nx": 100, "cfl": 0.4, "final_time": 1.08},
            {"nx": 400, "cfl": 0.4, "final_time": 1.08},
            r"Baseline: $N_x=100$",
            r"Refined: $N_x=400$",
        ),
    )
    for case_name, baseline, comparison, baseline_label, comparison_label in comparisons:
        print(f"\nGenerating {case_name}.mp4")
        comparison_results, video = run_parameter_comparison_example(
            baseline_parameters=baseline,
            comparison_parameters=comparison,
            baseline_label=baseline_label,
            comparison_label=comparison_label,
            output=output_directory / f"{case_name}.mp4",
            max_frames=90,
            fps=fps,
            dpi=dpi,
            preview=preview_each,
            download=False,
        )
        results[case_name] = comparison_results
        videos[case_name] = video

    archive = output_directory.with_suffix(".zip").resolve()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for video in videos.values():
            bundle.write(video, arcname=video.name)
    print(f"\nValidation archive saved to {archive}")
    if download_archive:
        download_generated_file(archive)
    return results, videos, archive


## 10. Optional command-line interface

`main(argv)` remains available if the notebook is exported to a Python script.
Inside Jupyter, call `main([])` to use parser defaults, or preferably use one of
the example functions above. The notebook deliberately does **not** execute
`raise SystemExit(main())`: in an IPython kernel that statement would display
the successful exit code as an exception.


In [11]:
def build_parser() -> argparse.ArgumentParser:
    """Build the command-line parser without reading process arguments."""
    parser = argparse.ArgumentParser(
        description="Solve the 1D inviscid Burgers equation with MacCormack."
    )
    parser.add_argument("--nx", type=int, default=400)
    parser.add_argument("--cfl", type=float, default=0.4)
    parser.add_argument("--final-time", type=float, default=1.10)
    parser.add_argument("--length", type=float, default=2.0)
    parser.add_argument("--plot", type=Path)
    parser.add_argument("--snapshot-study", type=Path)
    return parser


def main(argv: Sequence[str] | None = None) -> int:
    """Command-line entry point; pass ``[]`` when calling it in Jupyter."""
    args = build_parser().parse_args(argv)

    if args.snapshot_study is not None:
        run_snapshot_example(
            nx=args.nx,
            cfl=args.cfl,
            final_time=args.final_time,
            length=args.length,
            output=args.snapshot_study,
            show=False,
        )

    run_single_example(
        nx=args.nx,
        cfl=args.cfl,
        final_time=args.final_time,
        length=args.length,
        output=args.plot,
        show=False,
    )
    return 0


## 11. Extended animated validation campaign

The first six videos retain the validation structure of the advection
notebook, but replace comparisons among several linear schemes with stages of
the nonlinear evolution:

| Video | Purpose | Expected observation |
|---|---|---|
| 1. Early smooth transport | Initial reference | Small deformation and close agreement |
| 2. Nonlinear deformation | Solution-dependent speed | Left values catch the slower right values |
| 3. Strong steepening | Approach to breaking | Rapid growth of the spatial gradient |
| 4. Near breaking | Limit of smooth analysis | A very narrow transition layer |
| 5. Post-breaking run | Deliberate limitation case | Oscillations near the exact shock |
| 6. Reduced CFL | Time-step check | Smaller time step alone does not create a shock-capturing method |

Cases 1--4 remain before $t_{\mathrm b}=1$. Cases 5 and 6 clarify two
different issues: crossing the breaking time changes the mathematical regime,
whereas reducing the time step changes only the temporal resolution.


### Resolution and CFL experiments

Four synchronized comparison videos investigate parameters without changing
the numerical method:

| Video | Parameter change | What it demonstrates |
|---|---|---|
| 7. Smooth refinement | $N_x:100\rightarrow400$ at $t=0.70$ | Error decreases while the solution remains smooth |
| 8. CFL comparison | $C:0.4\rightarrow0.2$ at fixed grid | Effect of a smaller temporal step |
| 9. Near-breaking refinement | $N_x:100\rightarrow400$ at $t=0.98$ | The steep layer is increasingly demanding to resolve |
| 10. Post-breaking refinement | $N_x:100\rightarrow400$ at $t=1.08$ | Refinement does not turn MacCormack into a shock-capturing scheme |

The last video is diagnostic, not a recommendation to use MacCormack after
shock formation. Conservative high-resolution methods with appropriate
nonlinear stabilization are outside the scope of this chapter.


### Generate and download all ten videos

The following cell creates ten MP4 files and packages them in one ZIP archive.
In Google Colab, `download_archive=True` opens one download dialog. Set
`preview_each=True` to embed all ten videos; leaving it `False` avoids a very
large notebook output.


In [12]:
campaign_results, campaign_videos, campaign_archive = run_validation_campaign(
    output_directory="burgers_validation_videos",
    preview_each=False,
    download_archive=True,
    fps=20,
    dpi=100,
)



Generating 01_early_smooth_transport.mp4
method                  = conservative MacCormack
regime                  = smooth
grid cells              = 160
time steps              = 80
dt                      = 5.000000000000e-03
max-speed Courant no.   = 4.000000000000e-01
relative L2 error       = 3.506547632998e-03
mass error              = -3.272981493196e-05
numerical minimum       = 0.000000000000e+00
numerical maximum       = 1.007272235226e+00
Video saved to /content/burgers_validation_videos/01_early_smooth_transport.mp4 (81 frames, 20 fps).

Generating 02_nonlinear_deformation.mp4
method                  = conservative MacCormack
regime                  = smooth
grid cells              = 160
time steps              = 140
dt                      = 5.000000000000e-03
max-speed Courant no.   = 4.000000000000e-01
relative L2 error       = 7.051331801007e-03
mass error              = -3.272954969491e-05
numerical minimum       = 0.000000000000e+00
numerical maximum       = 1.012459

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Run a selected comparison animation

These are alternatives to the complete campaign and are therefore commented.


In [13]:
# Smooth regime, coarse versus refined grid:
# results_07, video_07 = run_parameter_comparison_example(
#     baseline_parameters={"nx": 100, "cfl": 0.4, "final_time": 0.70},
#     comparison_parameters={"nx": 400, "cfl": 0.4, "final_time": 0.70},
#     baseline_label=r"Baseline: $N_x=100$",
#     comparison_label=r"Refined: $N_x=400$",
#     output="07_smooth_grid_refinement.mp4",
#     preview=True,
#     download=True,
# )

# Near breaking, standard versus smaller CFL number:
# results_08, video_08 = run_parameter_comparison_example(
#     baseline_parameters={"nx": 200, "cfl": 0.4, "final_time": 0.90},
#     comparison_parameters={"nx": 200, "cfl": 0.2, "final_time": 0.90},
#     baseline_label=r"Baseline: $C=0.4$",
#     comparison_label=r"Smaller step: $C=0.2$",
#     output="08_cfl_comparison.mp4",
#     preview=True,
#     download=True,
# )


To download the archive again without recomputing or re-encoding the videos:


In [14]:
# download_generated_file("burgers_validation_videos.zip")
